# Credit-Card Transaction Anomaly Detection

A reproduction-ready experiment inspired by Kaggle's **Credit Card Fraud Detection** dataset. The workflow follows CRISP-DM and compares Isolation Forest, Local Outlier Factor (LOF), and One-Class SVM.

> **Execution disclaimer:** the source dataset is not distributed with this repository and this notebook has not been executed against it. All metric tables, anomaly counts, plots, and confusion matrices are produced only after a user supplies the data and executes the notebook. No numerical model result is claimed here.

## Reproduction checklist

1. Download `creditcard.csv` from the Kaggle Credit Card Fraud Detection dataset page after accepting its terms.
2. Put it in this notebook directory, or set `ANOMALY_DATA_PATH` to its location.
3. Create an environment with `python -m venv .venv`, activate it, and run `pip install -r requirements.txt`.
4. Launch `jupyter notebook anomaly_detection.ipynb` and run all cells from a fresh kernel.

Expected schema: anonymized numeric predictors `V1`–`V28`, `Time`, `Amount`, and binary target `Class` (1 denotes a labeled fraud). The loader validates the schema and prints setup help rather than crashing when the file is absent.

## 1. Business Understanding

**Objective.** Rank transactions for investigation using unsupervised anomaly detection, while using fraud labels only for retrospective evaluation—not for model fitting. This mirrors a setting in which confirmed fraud is scarce or delayed.

**Stakeholders and use.** Fraud operations can review flagged transactions; model-risk teams need reproducibility and understandable trade-offs. False positives create review cost and customer friction, while false negatives can create financial loss. Recall is therefore important, but precision and workload must also be monitored.

**Success criteria.** On supplied labeled data, compare precision, recall, F1, confusion matrices, flagged volume, and operational review capacity. Metrics are not pre-populated because they depend on actual execution. A production decision also requires cost-sensitive thresholds, temporal validation, latency testing, and stakeholder sign-off.

## 2. Data Understanding

The referenced public dataset represents card transactions with PCA-anonymized `V` features plus `Time`, `Amount`, and `Class`. Its strong class imbalance makes accuracy misleading. Labels are reserved for evaluation. This notebook deliberately does not download data automatically, avoiding hidden credentials, license ambiguity, and accidental use of a substitute dataset.

In [ ]:
from pathlib import Path
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, precision_recall_fscore_support
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import RobustScaler
from sklearn.svm import OneClassSVM

RANDOM_STATE = 42
DATA_PATH = Path(os.environ.get("ANOMALY_DATA_PATH", "creditcard.csv")).expanduser()
IMAGE_DIR = Path("images")
IMAGE_DIR.mkdir(exist_ok=True)
EXPECTED_FEATURES = ["Time", "Amount"] + [f"V{i}" for i in range(1, 29)]
TARGET = "Class"

DATA_AVAILABLE = DATA_PATH.is_file()
df = None
if not DATA_AVAILABLE:
    print(
        f"Dataset not found at {DATA_PATH.resolve()}.\n"
        "Download creditcard.csv from Kaggle's Credit Card Fraud Detection dataset, "
        "place it beside this notebook, or set ANOMALY_DATA_PATH. "
        "The remaining cells will skip data-dependent work safely."
    )
else:
    candidate = pd.read_csv(DATA_PATH)
    missing_columns = sorted(set(EXPECTED_FEATURES + [TARGET]) - set(candidate.columns))
    if missing_columns:
        warnings.warn(
            "Dataset was found but does not match the expected schema. "
            f"Missing columns: {missing_columns}. Data-dependent work is disabled."
        )
        DATA_AVAILABLE = False
    else:
        df = candidate[EXPECTED_FEATURES + [TARGET]].copy()
        print("Dataset loaded and schema validated. Continue to execute the analysis cells.")

## 3. Exploratory Data Analysis

The following cells reveal shape, types, distributions, class balance, and feature relationships only when the genuine file is available. Outputs are intentionally absent from the committed notebook.

In [ ]:
if DATA_AVAILABLE:
    display(df.head())
    display(df.dtypes.rename("dtype").to_frame())
    display(df.describe().T)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    df[TARGET].value_counts().sort_index().plot.bar(ax=axes[0], title="Observed class counts")
    axes[0].set_xlabel("Class")
    df["Amount"].plot.hist(bins=60, ax=axes[1], title="Amount distribution")
    df["Time"].plot.hist(bins=60, ax=axes[2], title="Time distribution")
    plt.tight_layout()
    plt.show()
else:
    print("EDA skipped: provide a valid dataset using the setup instructions above.")

## 4. Missing-value analysis

Count and percentage summaries locate incomplete fields. Missing predictors are median-imputed after splitting features from labels; a missing label cannot support evaluation and is removed. Median imputation is robust to extreme transaction values, though a missingness indicator may be preferable when missingness itself is informative.

In [ ]:
if DATA_AVAILABLE:
    missing = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percent": df.isna().mean().mul(100),
    }).sort_values("missing_count", ascending=False)
    display(missing)
else:
    print("Missing-value analysis skipped: dataset unavailable.")

## 5. Duplicate analysis

Exact duplicates may represent ingestion replay, but identical anonymized records might also be legitimate repeated events. This reproducible baseline reports duplicates and removes exact full-row copies. A production workflow should investigate transaction identifiers and timestamps before removal.

In [ ]:
if DATA_AVAILABLE:
    duplicate_mask = df.duplicated(keep=False)
    print("Exact duplicate rows (including all copies):", int(duplicate_mask.sum()))
    if duplicate_mask.any():
        display(df.loc[duplicate_mask].head())
else:
    print("Duplicate analysis skipped: dataset unavailable.")

## 6. Data cleaning

Cleaning enforces numeric fields, removes rows without a valid label, validates binary labels, drops exact duplicates, replaces infinite values, and median-imputes predictors. Assertions fail early with actionable messages when supplied data violate assumptions.

In [ ]:
if DATA_AVAILABLE:
    clean_df = df.drop_duplicates().copy()
    for column in EXPECTED_FEATURES + [TARGET]:
        clean_df[column] = pd.to_numeric(clean_df[column], errors="coerce")
    clean_df = clean_df.replace([np.inf, -np.inf], np.nan).dropna(subset=[TARGET])
    if not set(clean_df[TARGET].unique()).issubset({0, 1}):
        raise ValueError("Class must contain only binary values 0 and 1.")
    medians = clean_df[EXPECTED_FEATURES].median()
    if medians.isna().any():
        raise ValueError("At least one feature is entirely missing and cannot be median-imputed.")
    clean_df[EXPECTED_FEATURES] = clean_df[EXPECTED_FEATURES].fillna(medians)
    assert clean_df[EXPECTED_FEATURES + [TARGET]].notna().all().all()
    print("Cleaning complete. Inspect clean_df.shape after execution for retained rows.")
else:
    clean_df = None
    print("Cleaning skipped: dataset unavailable.")

## 7. Feature preparation

`Class` is excluded from training inputs to prevent label leakage. `log1p(Amount)` reduces positive skew, while `Time` is retained. All anonymized components remain available. Labels are kept solely for evaluation.

In [ ]:
if DATA_AVAILABLE:
    X_raw = clean_df[EXPECTED_FEATURES].copy()
    if (X_raw["Amount"] < 0).any():
        raise ValueError("Amount contains negative values; investigate before log transformation.")
    X_raw["Amount"] = np.log1p(X_raw["Amount"])
    y = clean_df[TARGET].astype(int).copy()
    feature_names = X_raw.columns.tolist()
    assert TARGET not in feature_names
else:
    X_raw = y = None
    feature_names = []

## 8. Feature scaling

Distance- and boundary-based algorithms are sensitive to scale. `RobustScaler` uses medians and interquartile ranges, limiting the influence of extremes. For a deployment-grade temporal experiment, fit transformations on an earlier training period and transform a later holdout period; this single-file demonstration fits the scaler without using labels.

In [ ]:
if DATA_AVAILABLE:
    scaler = RobustScaler()
    X_scaled = pd.DataFrame(
        scaler.fit_transform(X_raw), columns=feature_names, index=X_raw.index
    )
    assert np.isfinite(X_scaled.to_numpy()).all()
    print("Feature matrix prepared and robust-scaled.")
else:
    X_scaled = None
    print("Scaling skipped: dataset unavailable.")

## 9. Outlier analysis

Univariate IQR flags are diagnostic—not ground truth and not automatically removed. Extreme values may be the very cases of interest. The plot and summary below are generated only from supplied data.

In [ ]:
if DATA_AVAILABLE:
    q1, q3 = X_raw.quantile(0.25), X_raw.quantile(0.75)
    iqr = q3 - q1
    iqr_flags = X_raw.lt(q1 - 1.5 * iqr) | X_raw.gt(q3 + 1.5 * iqr)
    display(iqr_flags.sum().sort_values(ascending=False).rename("IQR-flagged rows").to_frame())

    X_raw[["Time", "Amount"]].plot.box(subplots=True, figsize=(10, 4), title="Univariate outlier diagnostics")
    plt.tight_layout()
    plt.show()
else:
    print("Outlier diagnostics skipped: dataset unavailable.")

## 10–12. Models and contamination parameter

### Contamination
`contamination` is the assumed fraction of observations to flag. It is a **decision parameter, not an estimate of fraud prevalence**. Here it defaults to `"auto"` so Isolation Forest and LOF derive their standard score offset rather than borrowing the true label rate. Labels are never used to set it. In practice, tune a numeric fraction using review capacity and validation data, and report sensitivity across plausible values. Changing contamination changes the threshold and flagged count, so no result is claimed without execution.

### 10. Isolation Forest
Random partitions isolate sparse, unusual points quickly. It scales well and handles nonlinear multivariate structure, but its random partitions and threshold can be hard to explain.

### 11. Local Outlier Factor
LOF compares local density with neighboring density and can find anomalies in clusters of varying density. It is sensitive to scaling and `n_neighbors`, and fitting/prediction can become expensive on large data.

### 12. One-Class SVM
An RBF boundary can model flexible normal regions. It is sensitive to scale, `nu`, and `gamma`, and its time/memory cost may be high. `nu` is an upper bound on training errors and lower bound on support-vector fraction; it is not exactly the achieved anomaly fraction.

In [ ]:
if DATA_AVAILABLE:
    contamination = "auto"  # Change only as an explicitly documented operational assumption.
    n_neighbors = min(20, len(X_scaled) - 1)
    if n_neighbors < 2:
        raise ValueError("At least three cleaned rows are required for LOF.")

    models = {
        "Isolation Forest": IsolationForest(
            n_estimators=200, contamination=contamination, random_state=RANDOM_STATE, n_jobs=-1
        ),
        "Local Outlier Factor": LocalOutlierFactor(
            n_neighbors=n_neighbors, contamination=contamination, n_jobs=-1
        ),
        "One-Class SVM": OneClassSVM(kernel="rbf", gamma="scale", nu=0.01),
    }
    predictions, anomaly_scores = {}, {}

    iso_raw = models["Isolation Forest"].fit_predict(X_scaled)
    predictions["Isolation Forest"] = (iso_raw == -1).astype(int)
    anomaly_scores["Isolation Forest"] = -models["Isolation Forest"].decision_function(X_scaled)

    lof_raw = models["Local Outlier Factor"].fit_predict(X_scaled)
    predictions["Local Outlier Factor"] = (lof_raw == -1).astype(int)
    anomaly_scores["Local Outlier Factor"] = -models["Local Outlier Factor"].negative_outlier_factor_

    svm_raw = models["One-Class SVM"].fit_predict(X_scaled)
    predictions["One-Class SVM"] = (svm_raw == -1).astype(int)
    anomaly_scores["One-Class SVM"] = -models["One-Class SVM"].decision_function(X_scaled).ravel()
    print("All three unsupervised models fitted; labels were not used during fitting.")
else:
    models, predictions, anomaly_scores = {}, {}, {}
    print("Model fitting skipped: dataset unavailable.")

## 13, 15–17. Comparison and labeled evaluation

Where binary labels exist, the cell computes anomaly count, precision, recall, and F1 for each technique. It then plots a confusion matrix containing true negatives, false positives, false negatives, and true positives. `zero_division=0` makes empty-positive predictions explicit rather than crashing. Accuracy is omitted because severe imbalance can make it misleading.

The table and matrices below are intentionally empty in this committed notebook; they become empirical results only when run on the supplied data.

In [ ]:
if DATA_AVAILABLE:
    rows = []
    for name, predicted in predictions.items():
        precision, recall, f1, _ = precision_recall_fscore_support(
            y, predicted, average="binary", zero_division=0
        )
        rows.append({
            "model": name,
            "flagged_anomalies": int(predicted.sum()),
            "precision": precision,
            "recall": recall,
            "f1": f1,
        })
    comparison = pd.DataFrame(rows).set_index("model")
    display(comparison.style.format({"precision": "{:.4f}", "recall": "{:.4f}", "f1": "{:.4f}"}))

    fig, axes = plt.subplots(1, len(predictions), figsize=(15, 4))
    for ax, (name, predicted) in zip(axes, predictions.items()):
        ConfusionMatrixDisplay(confusion_matrix(y, predicted), display_labels=["Normal", "Fraud"]).plot(
            ax=ax, colorbar=False, values_format="d"
        )
        ax.set_title(name)
    plt.tight_layout()
    plt.savefig(IMAGE_DIR / "confusion_matrices.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    comparison = pd.DataFrame()
    print("Evaluation skipped: labels and model predictions require the dataset.")

## 18. Anomaly visualization

PCA is used only to project scaled features into two dimensions for visualization; models were fitted in the full feature space. Overlap in a 2-D projection does not prove that a model is wrong. Each generated point reflects real supplied data and model output.

In [ ]:
if DATA_AVAILABLE:
    projection = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X_scaled)
    fig, axes = plt.subplots(1, len(predictions), figsize=(16, 4), sharex=True, sharey=True)
    for ax, (name, predicted) in zip(axes, predictions.items()):
        ax.scatter(projection[:, 0], projection[:, 1], c=predicted, cmap="coolwarm", s=7, alpha=0.55)
        ax.set_title(name)
        ax.set_xlabel("PCA component 1")
    axes[0].set_ylabel("PCA component 2")
    plt.tight_layout()
    plt.savefig(IMAGE_DIR / "anomaly_projection.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Anomaly visualization skipped: dataset unavailable.")

## 19. Interpretation of detected anomalies

The next cell ranks transactions by each detector's anomaly score and displays their feature values and labels for audit. High scores mean only that a transaction is unusual under that model—not that fraud is proven. Compare the most influential deviations, agreement across models, transaction amount/time, and eventual labels. Because `V` features are anonymized PCA components, causal or business interpretation is inherently limited.

In [ ]:
if DATA_AVAILABLE:
    interpretation_tables = {}
    for name, scores in anomaly_scores.items():
        ranked = clean_df.loc[:, EXPECTED_FEATURES + [TARGET]].copy()
        ranked["anomaly_score"] = scores
        ranked["flagged"] = predictions[name]
        interpretation_tables[name] = ranked.sort_values("anomaly_score", ascending=False).head(10)
        print(f"Top-ranked observations: {name}")
        display(interpretation_tables[name])
else:
    print("Interpretation tables skipped: dataset unavailable.")

## 20. Strengths and weaknesses

| Technique | Strengths | Weaknesses |
|---|---|---|
| Isolation Forest | Efficient; nonlinear; few distributional assumptions; reproducible with a seed | Threshold-dependent; unstable for some feature representations; limited case-level explanation |
| Local Outlier Factor | Detects local density deviations; useful with heterogeneous clusters | Scale/neighborhood sensitive; costly at prediction scale; standard mode is transductive |
| One-Class SVM | Flexible nonlinear boundary; strong on compact normal regions | Sensitive to hyperparameters and scaling; computationally demanding; `nu` is easily misinterpreted |

No universal winner should be inferred without execution and a metric/cost objective. Agreement can strengthen triage confidence, while disagreement identifies cases for investigation.

## 21. Limitations

- The original dataset is not bundled or executed, so this artifact reports no empirical outcome.
- Fraud labels can be delayed, incomplete, or biased toward previously investigated behavior.
- The anonymized features limit domain interpretation and fairness analysis.
- Random or full-data evaluation may overstate future performance under concept drift; temporal holdouts are preferable.
- Unsupervised anomalies are not synonymous with fraud, and ordinary but rare transactions may be flagged.
- Hyperparameters and thresholds have not been tuned, calibrated to investigation cost, or stress-tested here.
- Median imputation and duplicate removal are baseline choices that require domain validation.

## 22. Future improvements

1. Use chronological train/validation/test windows and monitor drift.
2. Tune contamination, `n_neighbors`, `nu`, and `gamma` against review capacity and cost-weighted validation objectives.
3. Add precision–recall curves, precision at *k*, recall at a fixed alert budget, and bootstrap confidence intervals.
4. Compare robust covariance, autoencoders, and supervised or semi-supervised baselines when labels permit.
5. Engineer behavioral, velocity, merchant, and account-history features without leaking future information.
6. Add explainability, analyst feedback, threshold governance, fairness checks, and production monitoring.

## 23. CRISP-DM conclusion

- **Business understanding:** the task is alert prioritization under asymmetric fraud and review costs.
- **Data understanding:** schema checks, EDA, missingness, duplicates, imbalance, and outlier diagnostics are implemented.
- **Data preparation:** numeric coercion, conservative cleaning, leakage-free feature selection, transformation, and robust scaling are reproducible.
- **Modeling:** Isolation Forest, LOF, and One-Class SVM are implemented without training on `Class`.
- **Evaluation:** labeled precision, recall, F1, confusion matrices, score ranking, and visual comparison run only when real data exist.
- **Deployment:** thresholds must reflect capacity and costs, with temporal validation, monitoring, and analyst feedback.

The workflow is complete but intentionally makes no numerical claim until the dataset is supplied and the notebook is executed. CRISP-DM is iterative: evaluation findings should return the team to business objectives, data quality, preparation, and modeling choices.